### Model Parameter 설정

In [3]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:5])


sk-pr


### 1) 모델 파라미터 설정
##### Temperature 효과
* 0.2 (낮은 값): 더 예측 가능하고 일반적인 이야기
* 1.2 (높은 값): 새로운 요소가 등장하며, 더 창의적이고 흥미로운 이야기 생성

##### Presence Penalty 효과
* 0.0 (낮은 값): 기존에 자주 등장하는 단어와 구조 유지
* 1.5 (높은 값): 새로운 표현과 독창적인 아이디어 등장

In [ ]:
from langchain_openai import ChatOpenAI

# 보수적인 설정 (일관되고 예측 가능한 답변)
llm_conservative = ChatOpenAI(    
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",    
    temperature=0.1,  # 매우 낮은 온도 - 예측 가능하고 일관된 출력
    presence_penalty=-0.5,  # 기존 패턴과 표현을 강하게 선호
    frequency_penalty=-0.3,  # 반복적 표현 허용, 일관성 증대
    max_tokens=350,  # 적당한 길이
    top_p=0.3  # 매우 제한적인 토큰 선택 - 안전하고 예측 가능한 단어만
)

# 창의적인 설정 (독창적이고 예측 불가능한 답변)
llm_creative = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini", 
    temperature=1.5,  # 매우 높은 온도 - 창의적이고 예측 불가능한 출력
    presence_penalty=1.5,  # 새로운 주제와 개념을 강하게 유도
    frequency_penalty=1.2,  # 반복을 강하게 억제하여 매우 다양한 표현 생성
    max_tokens=350,  # 더 긴 창의적 서술 허용
    top_p=0.95  # 거의 모든 토큰 후보에서 선택 - 최대 창의성
)

# 질문 설정: 파라미터 차이가 극명하게 드러나는 추상적/철학적 질문
question = """
재미있고 흥미로운 짧은 동화를 만들어 주세요. 
동화의 주제는 "신데렐라"입니다.
"""

print("=" * 60)
print(" CONSERVATIVE 설정 (온도:0.1, presence:-0.5, frequency:-0.3, top_p:0.3)")
print("  → 전통적이고 학술적인 철학적 접근")
print("=" * 60)

# 보수적 모델 호출
response_conservative = llm_conservative.invoke(question)
print(response_conservative.content)

print("\n" + "=" * 60)
print(" CREATIVE 설정 (온도:1.5, presence:1.5, frequency:1.2, top_p:0.95)")
print("  → 추상적이고 독창적인 철학적 사고")
print("=" * 60)

# 창의적 모델 호출
response_creative = llm_creative.invoke(question)
print(response_creative.content)

print("\n" + "=" * 60)
print(" 파라미터 차이점 요약:")
print("   • Temperature: 0.1 vs 1.5 (15배 차이)")
print("   • Presence Penalty: -0.5 vs 1.5 (기존 패턴 선호 vs 새로운 개념 강요)")
print("   • Frequency Penalty: -0.3 vs 1.2 (반복 허용 vs 강한 다양성)")
print("   • Top-p: 0.3 vs 0.95 (제한적 vs 거의 무제한 토큰 선택)")
print("=" * 60)

#### [추가] 같은 질문을 여러 번 실행해 파라미터 효과 확인하기

위 예제는 설정별로 **한 번씩만** 실행하므로 차이가 뚜렷하게 보이지 않을 수 있습니다. 이유는 다음과 같습니다.

- **줄거리가 정해진 질문**: 신데렐라는 널리 알려진 이야기라 어떤 설정에서도 비슷한 줄거리를 따라갑니다.
- **1회 비교의 한계**: 한 번만 비교하면 차이가 설정 때문인지 우연인지 구분할 수 없습니다.
- **응답이 중간에 잘림**: `max_tokens=350`에서 두 응답이 같은 지점에서 끊겨 뒷부분의 차이가 보이지 않습니다.

`temperature` 같은 파라미터는 응답 하나를 정해 주는 값이 아니라 모델이 **답을 뽑는 확률 분포의 폭**을 조절합니다. 그래서 **같은 질문을 여러 번 실행해서 응답이 얼마나 다양한지** 비교해야 효과가 드러납니다.

| 설정 | 같은 질문을 여러 번 실행하면 |
|---|---|
| `temperature` 낮음 | 거의 같은 응답이 반복됨 |
| `temperature` 높음 | 매번 다른 응답이 나옴 |

아래 코드는 이 점을 확인하도록 **원래 코드를 수정한 버전**입니다.

- 같은 질문을 5회(또는 3회) 반복하고 **"서로 다른 응답 수"** 를 출력합니다.
- 정답이 하나가 아닌 짧은 질문(이름 짓기, 새로운 배경의 첫 문장)을 사용합니다.
- `top_p`는 `temperature`와 같은 계열의 옵션이라 기본값으로 두고, `temperature`와 페널티만 바꿉니다.
- 여러 번 호출하므로 **비용 부담이 적은 Groq(`GROQ_API_KEY`)** 를 사용합니다.

> 반복 실행은 효과를 **관찰하기 위한 실습용 방법**입니다. 실제 서비스에서는 한 번만 요청하며, 일관된 답이 필요하면 `temperature`를 낮추고 다양한 답이 필요하면 높입니다.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# 여러 번 호출하므로 비용 부담이 적은 Groq(OpenAI 호환 엔드포인트)를 사용합니다.
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-20b"

# 같은 모델에서 temperature(와 페널티)만 다르게 비교합니다.
# top_p 는 temperature 와 같은 계열의 옵션이라, 함께 바꾸면 어떤 효과인지 구분하기 어려워 기본값(1.0)으로 둡니다.
COMMON = dict(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL,
    model=GROQ_MODEL,
    max_tokens=150,
    max_retries=5,  # 429(분당 한도), 503 등 일시 오류 자동 재시도
)

# 보수적 설정: 확률이 가장 높은 단어 위주로 선택 → 실행할 때마다 거의 같은 결과
llm_conservative = ChatOpenAI(temperature=0.0, **COMMON)

# 창의적 설정: 확률이 낮은 단어도 선택하고, 이미 쓴 표현·주제는 피하도록 유도 → 실행할 때마다 다른 결과
llm_creative = ChatOpenAI(temperature=1.2, presence_penalty=0.6, frequency_penalty=0.3, **COMMON)


def run_many(llm, question: str, n: int) -> list[str]:
    """같은 질문을 n번 실행해 응답 문자열 리스트를 반환합니다. (무료 한도를 고려해 동시 실행 수 제한)"""
    results = llm.batch([question] * n, config={"max_concurrency": 2})
    return [r.content.strip() for r in results]


def show(title: str, outputs: list[str]) -> None:
    """응답 목록과 '서로 다른 응답 수'를 출력합니다."""
    print(f"[{title}] 서로 다른 응답 수: {len(set(outputs))} / {len(outputs)}")
    for i, o in enumerate(outputs, 1):
        print(f"  {i}. {o}")
    print()


# 실험 1: 정답이 하나가 아닌 짧은 질문 (5회)
question1 = "신데렐라 이야기에 새로 등장할 마법사 캐릭터의 이름을 하나만 지어 주세요. 이름만 출력하세요."

print("=" * 70)
print(f" 실험 1) {question1}")
print("=" * 70)
show("보수적: temperature=0.0", run_many(llm_conservative, question1, 5))
show("창의적: temperature=1.2, presence=0.6, frequency=0.3", run_many(llm_creative, question1, 5))

# 실험 2: 같은 이야기를 완전히 새로운 배경으로 바꾸는 질문 (3회)
question2 = "신데렐라 이야기를 지금까지 없던 새로운 시대와 배경으로 바꿔서, 동화의 첫 두 문장만 써 주세요."

print("=" * 70)
print(f" 실험 2) {question2}")
print("=" * 70)
show("보수적: temperature=0.0", run_many(llm_conservative, question2, 3))
show("창의적: temperature=1.2, presence=0.6, frequency=0.3", run_many(llm_creative, question2, 3))

print("=" * 70)
print(" 기대 결과")
print("   • 보수적 설정: 같은 질문에 거의 같은 응답이 반복됨 (서로 다른 응답 수가 적음)")
print("   • 창의적 설정: 실행할 때마다 다른 이름·배경이 등장함 (서로 다른 응답 수가 많음)")
print("=" * 70)

#### Model Parameter
* Temperature (온도): 이 파라미터는 모델의 예측 불확실성을 조절합니다. 
    * 낮은 온도는 모델이 가장 확률이 높은 다음 단어를 선택하게 하여 안정적인 결과를 내고, 높은 온도는 확률이 낮은 단어까지 선택지에 포함시켜 더 창의적이고 예측 불가능한 답변을 생성합니다. 
* Presence Penalty & Frequency Penalty: 두 파라미터 모두 토큰(단어)의 반복성을 제어합니다.
    * Presence Penalty는 이전에 나온 단어나 주제에 '페널티'를 주어 새로운 단어를 사용하게 합니다. 값이 높을수록 새로운 내용을 더 자주 시도합니다.
    * Frequency Penalty는 단어가 등장한 빈도에 비례하여 '페널티'를 줍니다. 값이 높을수록 특정 단어를 반복적으로 사용하는 것을 강하게 억제하여 더 다양한 어휘를 사용하게 합니다. 
* Top_p: 이 파라미터는 다음 토큰을 선택할 때 고려할 단어의 범위를 조절합니다. Top_p가 0.3이면 확률이 높은 상위 30%의 단어 중에서만 선택하고, 0.95면 확률이 낮은 단어들까지 포함하여 거의 모든 단어를 고려합니다. Top_p가 높을수록 모델의 선택지가 넓어져 더 자유로운 응답이 가능합니다.    

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 보수적인 설정: 논리적이고 일관된 서술
llm_conservative = ChatOpenAI(
    api_key=GROQ_API_KEY, 
    base_url="https://api.groq.com/openai/v1",
    model="openai/gpt-oss-20b",
    #model="moonshotai/kimi-k2-instruct-0905", 
    temperature=0.1,         # 낮은 값(0.1): '확률의 폭'을 좁혀 가장 안전하고 흔한 단어를 선택합니다.
                             # → 결과가 예측 가능하고 반복적일 수 있습니다. 마치 정해진 교과서처럼.
    presence_penalty=-0.5,   # 음수 값(-0.5): 이미 등장했던 주제나 단어를 다시 사용하는 것을 권장합니다.
                             # → 기존 패턴을 벗어나지 않고 일관성을 유지합니다.
    frequency_penalty=-0.3,  # 음수 값(-0.3): 자주 등장한 단어에 대한 '벌칙'을 줄여, 동일한 표현의 반복을 허용합니다.
                             # → 응답의 일관성이 높아지고, 핵심 주제에서 벗어나지 않습니다.
    max_tokens=200,          # 응답의 최대 길이를 제한합니다. 
    top_p=0.3                # 낮은 값(0.3): 확률이 높은 상위 30% 토큰(단어)만 고려합니다.
                             # → 모델의 '선택지'를 극도로 제한하여 예측 가능한 단어만 사용합니다.
)

# 창의적인 설정: 독창적이고 풍부한 비유와 은유
llm_creative = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model="openai/gpt-oss-20b",
    temperature=1.5,         # 높은 값(1.5): '확률의 폭'을 넓혀 흔치 않은 단어까지 선택하게 합니다.
                             # → 예측 불가능하고 독창적인 문장을 생성합니다. 마치 즉흥적인 시인처럼요.
    presence_penalty=1.5,    # 양수 값(1.5): 이미 사용된 주제나 개념에 '벌칙'을 주어 새로운 것을 시도하게 합니다.
                             # → 독창적인 아이디어나 관점을 강하게 유도합니다.
    frequency_penalty=1.2,   # 양수 값(1.2): 자주 등장한 단어에 '벌칙'을 주어 반복을 강력하게 억제합니다.
                             # → 표현의 다양성을 극대화하고 문장이 풍부해집니다.
    max_tokens=350,          # 더 긴 창의적 서술을 허용하기 위해 최대 길이를 늘립니다.
    top_p=0.95               # 높은 값(0.95): 확률이 낮은 95%의 모든 토큰을 고려합니다.
                             # → 모델의 '선택지'를 거의 무제한으로 확장하여 최대의 창의성을 발휘합니다.
)

# 질문을 ChatPromptTemplate으로 정의하여 역할과 질문을 명확히 함
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 창의적인 작가입니다. 주어진 주제에 대해 풍부한 비유와 은유를 사용하여 서술해주세요."),
    ("human", """
    "도시의 소음"을 생명체나 자연 현상에 빗대어 설명해주세요.
    일상적인 소리가 아닌, 그 소리가 가진 생명력과 감정을 중심으로 묘사해주세요.
    """),
])

# 파서 설정
parser = StrOutputParser()

# 체인 구성: Prompt -> Model -> Parser
chain_conservative = prompt | llm_conservative | parser
chain_creative = prompt | llm_creative | parser

# 모델 호출 및 결과 출력
print("=" * 60)
print(" 보수적 설정 (논리적, 일관적)")
print(" → 일상적이고 일반적인 비유를 사용하여 안정적인 답변")
print("=" * 60)
response_conservative = chain_conservative.invoke({})
print(response_conservative)

print("\n" + "=" * 60)
print(" 창의적 설정 (풍부한 비유와 은유)")
print(" → 예상치 못한 독창적인 비유와 감각적인 표현 사용")
print("=" * 60)
response_creative = chain_creative.invoke({})
print(response_creative)

In [ ]:
# 질문을 담은 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주어진 주제에 대해 풍부한 비유와 은유를 사용하여 서술하는 전문가입니다. 답변의 관점을 명확히 구분하여 설명해주세요."),
    ("human", """
    "빨간 장미"를 두 가지 관점에서 묘사해주세요.
    1. **객관적 관점:** 장미의 물리적 특성과 상징적 의미를 중심으로 설명해주세요.
    2. **문학적 관점:** 장미를 사람이나 감정에 빗대어 자유롭고 시적으로 묘사해주세요.
    """),
])

# 파서 설정
parser = StrOutputParser()

# 체인 구성: Prompt -> Model -> Parser
chain_conservative = prompt | llm_conservative | parser
chain_creative = prompt | llm_creative | parser

# 모델 호출 및 결과 출력
print("=" * 60)
print(" 보수적 설정 (객관적, 사실 기반)")
print(" → 과학적이고 논리적인 장미의 특성 설명")
print("=" * 60)
response_conservative = chain_conservative.invoke({})
print(response_conservative)

print("\n" + "=" * 60)
print(" 창의적 설정 (문학적, 상상 기반)")
print(" → 은유와 비유를 활용한 장미의 감성적 묘사")
print("=" * 60)
response_creative = chain_creative.invoke({})
print(response_creative)

##### 2-1) 모델에 직접 파라미터를 전달

`ChatOpenAI(model=..., temperature=..., max_tokens=...)`처럼 **모델을 만들 때 파라미터를 넘기면**, 그 모델로 보내는 모든 요청에 적용됩니다. 파라미터의 효과를 분명하게 보려면 **한 번에 하나만 바꿔서** 비교해야 합니다.

| 비교 | 바꾸는 값 (나머지는 동일) | 확인할 점 |
|---|---|---|
| ① 응답 길이 | `max_tokens` 300 vs 1500 | 300: 일정이 중간에 끊김 (`finish_reason='length'`) / 1500: 4일 일정이 끝까지 완성됨 (`finish_reason='stop'`) |
| ② 응답 다양성 | `temperature` 0.2 vs 1.2 (같은 질문을 여러 번 실행) | 정답이 여러 개일 수 있는 질문에서, 낮음은 비슷한 응답이 반복되고 높음은 실행할 때마다 다른 응답이 나옴 |

> - `max_tokens`는 응답 길이의 **상한**입니다. 상한에 걸리면 응답이 문장 중간에서 잘립니다.
> - `temperature`의 효과는 한 번이 아니라 **여러 번 실행한 응답의 다양성**으로 드러납니다.
> - 이 비교는 Groq의 `openai/gpt-oss-120b`로 실행합니다. 이 모델은 **추론 모델이라 추론 토큰도 `max_tokens`에 포함**되어, 상한이 작으면 본문이 비어 나올 수 있습니다. 그래서 `reasoning_effort="low"`로 추론량을 줄였습니다.
> - ①의 질문은 표·머리말·여행 팁을 빼고 "각 두 문장씩"으로 제한했습니다. 제한이 없으면 응답 길이가 실행마다 크게 달라져 1500 토큰에서도 잘릴 수 있습니다.
> - 여러 번 호출하므로 비용 부담이 적은 Groq(`GROQ_API_KEY`)를 사용합니다.
> - `frequency_penalty`, `presence_penalty`는 효과가 작고 모델에 따라 적용 여부가 달라 이 비교에서는 제외했습니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# GROQ_API_KEY, GROQ_BASE_URL 은 위 [추가] 셀에서 정의한 값을 사용합니다.
# gpt-oss 는 추론 모델이라 추론 토큰도 max_tokens 에 포함됩니다.
# reasoning_effort="low" 로 추론량을 줄여, max_tokens 가 작아도 본문이 나오도록 합니다.
PARAM_TEST_MODEL = "openai/gpt-oss-120b"


def make_llm(**params) -> ChatOpenAI:
    """Groq 모델을 만들고, 전달한 파라미터를 모델 생성 시점에 적용합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=PARAM_TEST_MODEL,
        reasoning_effort="low",  # 추론 토큰 최소화 (max_tokens 비교를 분명하게 하기 위함)
        max_retries=5,           # 429(분당 한도), 503 등 일시 오류 자동 재시도
        **params,
    )


# ① 응답 길이 비교: max_tokens 만 다르게 하고 나머지는 동일
short_model = make_llm(temperature=0.7, max_tokens=300)    # 짧은 상한
long_model = make_llm(temperature=0.7, max_tokens=1500)    # 넉넉한 상한

# 프롬프트 구성
system_message = SystemMessagePromptTemplate.from_template(
    "당신은 여행 전문가입니다. 사용자의 요청에 맞는 최적의 여행지를 추천해 주세요."
)
user_message = HumanMessagePromptTemplate.from_template("{user_input}")
chat_prompt = ChatPromptTemplate.from_messages([system_message, user_message])

# 체인 생성 (프롬프트와 질문은 같고, 모델의 max_tokens 만 다름)
short_chain = chat_prompt | short_model
long_chain = chat_prompt | long_model

# 일자별 일정 전체가 필요한 질문이라, 상한이 작으면 끝까지 쓰지 못해 차이가 드러남
# (표·머리말·여행 팁을 제외해 응답 길이를 약 600~800 토큰으로 맞춤: 300 에서는 잘리고 1500 에서는 완성됨)
test_question = (
    "가족과 함께 3박 4일 동안 한국에서 여유롭게 여행할 수 있는 일정을 동선을 고려하여 추천해 주세요. "
    "표, 머리말, 여행 팁 없이 1일차부터 4일차까지 일자별로 오전·오후·저녁 일정을 각각 두 문장씩(총 24문장)만 출력하세요."
)

short_response = short_chain.invoke({"user_input": test_question})
long_response = long_chain.invoke({"user_input": test_question})


def show_result(title: str, response) -> None:
    """응답 본문과 종료 사유, 출력 토큰 수를 출력합니다."""
    finish = response.response_metadata.get("finish_reason")   # 'length' 면 max_tokens 상한에 걸려 잘린 것
    tokens = response.usage_metadata["output_tokens"]
    print(f" [{title}] finish_reason={finish}, 출력 토큰={tokens}")
    print(response.content)
    print("\n" + "=" * 80 + "\n")


show_result("max_tokens=300", short_response)
show_result("max_tokens=1500", long_response)

##### 2-1-2) 응답 다양성 비교 (`temperature`)

같은 질문을 **여러 번(5회)** 실행하고 응답이 얼마나 달라지는지 비교합니다. `temperature`는 응답 하나를 정하는 값이 아니라 **답을 뽑는 확률 분포의 폭**을 조절하므로, 한 번이 아니라 반복 실행의 결과로 효과를 확인합니다.

질문을 **두 가지**로 나눈 이유는 `temperature`가 **언제 효과가 있는지**까지 확인하기 위해서입니다.

| 질문 | 성격 | 기대 결과 |
|---|---|---|
| 질문 1: 추천 지역 이름만 출력 | 답이 사실상 하나로 정해짐 (대조군) | `temperature`를 높여도 같은 지역이 반복됨 |
| 질문 2: 여행 코스 제목 짓기 | 정답이 여러 개일 수 있음 | 0.2는 거의 같은 제목, 1.2는 매번 다른 제목 |

- **읽는 법**: `서로 다른 응답 수`가 작으면 응답이 일관되고, 크면 응답이 다양하다는 뜻입니다. 응답이 길면 문장 표현만 달라도 "다른 응답"으로 세어지므로, **이름이나 제목처럼 짧은 답**으로 비교합니다.
- **핵심**: `temperature`는 **답의 후보가 여러 개일 때** 효과가 큽니다. 정답이 정해진 질문에서는 값을 올려도 거의 달라지지 않습니다.
- **활용 기준**: 일관된 답이 필요한 작업(분류, 추출)은 `temperature`를 낮추고, 다양한 아이디어가 필요한 작업(이름, 제목, 글쓰기)은 높입니다.
- 반복 실행은 효과를 **관찰하기 위한 실습용 방법**입니다. 실제 서비스에서는 한 번만 요청합니다.

In [ ]:
# ② 응답 다양성 비교: temperature 만 다르게 하고, 같은 질문을 여러 번 실행
# make_llm() 은 위 셀, run_many() / show() 는 위 [추가] 실험 셀에서 정의한 함수를 사용합니다.
# 추론 토큰도 max_tokens 에 포함되므로, 짧은 답변이라도 여유 있게 300 으로 설정합니다.
low_model = make_llm(temperature=0.2, max_tokens=300)     # 낮은 temperature: 확률이 높은 답 위주
high_model = make_llm(temperature=1.2, max_tokens=300)    # 높은 temperature: 확률이 낮은 답도 선택

# 질문 1 (대조군): 답이 사실상 하나로 정해지는 질문 → temperature 를 높여도 응답이 거의 달라지지 않음
question_fixed = "가족과 함께 3박 4일 한국 여행을 간다면 추천하는 여행 지역 한 곳의 이름만 출력하세요."

# 질문 2: 정답이 여러 개일 수 있는 창의적인 질문 → temperature 에 따라 다양성이 크게 달라짐
question_creative = (
    "가족과 함께 떠나는 3박 4일 한국 여행 코스에 붙일 창의적인 제목을 하나만 지어 주세요. 제목만 출력하세요."
)

for label, question in [("질문 1 (정답이 사실상 하나)", question_fixed), ("질문 2 (정답이 여러 개)", question_creative)]:
    print("=" * 80)
    print(f" {label}: {question}")
    print("=" * 80)
    show("temperature=0.2", run_many(low_model, question, 5))
    show("temperature=1.2", run_many(high_model, question, 5))

print("=" * 80)
print(" 기대 결과")
print("   • 질문 1: 두 설정 모두 같은 지역이 반복됨 → 정답이 하나로 정해지는 질문은 temperature 의 영향이 작음")
print("   • 질문 2: temperature=0.2 는 거의 같은 제목, 1.2 는 매번 다른 제목 → 답의 후보가 여럿일 때 효과가 큼")
print("=" * 80)

#### 2-2) 모델에 추가적인 파라미터를 전달
- bind() 메서드를 사용
- max_tokens=50 (기본) / max_tokens=150 (상세한 설명) 
- stop=["."] → 첫 번째 마침표에서 응답을 중단
- temperature=0.8 → 보다 창의적인 답변을 생성함

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#  프롬프트 설정 (천문학 질문에 대한 답변을 생성하는 시스템)
prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 천문학 질문에 대해 명확하고 자세한 답변을 제공할 수 있습니다."),
    ("user", "{user_input}"),
])

#  기본 모델 설정 (기본적인 답변 생성)
base_model = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    max_tokens=150
)

#  질문 설정
# 1. MAX_TOKENS 차이를 보여주는 질문 (길이 제한 효과)
max_tokens_question = "인공지능의 발전이 미래 사회에 미칠 영향을 긍정적 측면과 부정적 측면으로 나누어 자세히 설명해 주세요."

# 2. STOP 파라미터 차이를 보여주는 질문 (중단점 효과)
stop_question = "Python 프로그래밍을 배우는 초보자에게 추천하는 학습 단계를 순서대로 설명해 주세요. 각 단계별로 구체적인 방법과 팁을 포함해서 답변해 주세요."

# 3. TEMPERATURE 차이를 보여주는 질문 (창의성 vs 정확성)
temperature_question = "시간 여행이 가능하다면 과거의 어느 시대로 가고 싶은지와 그 이유를 창의적으로 설명해 주세요."

# 4. 복합적 비교를 위한 질문 (모든 파라미터 효과)
complex_question = "화성에 인류가 정착하기 위해 필요한 기술과 준비사항들을 단계별로 설명하고, 각 단계에서 예상되는 도전과제와 해결방안을 제시해 주세요."

question = stop_question

#  Before (기본 max_tokens=150)
messages = prompt.format_messages(user_input=question)
before_answer = base_model.invoke(messages)

#  Before 출력
print("\n [Before] 기본 max_tokens=150 (기본 답변)")
print(before_answer.content)

print("\n" + "="*100 + "\n")  # 가독성을 위한 구분선

#  After (파라미터 조정 후 비교)
stop_chain = prompt | base_model.bind(max_tokens=150, stop=["."])  # 첫 번째 마침표에서 중단
temp_chain = prompt | base_model.bind(max_tokens=150, temperature=0.8)  # 창의적인 답변 유도

stop_answer = stop_chain.invoke({"user_input": question})
temp_answer = temp_chain.invoke({"user_input": question})

#  After 출력 (stop vs temperature 비교)
print(" [After] max_tokens=150, stop=['.'] (첫 번째 마침표에서 응답 중단)")
print(stop_answer.content)

print("\n" + "="*100 + "\n")  # 가독성을 위한 구분선

print(" [After] max_tokens=150, temperature=0.8 (창의적인 답변)")
print(temp_answer.content)


 [Before] 기본 max_tokens=150 (기본 답변)
## Python 입문자를 위한 **학습 로드맵**  
아래 순서대로 진행하면 “문법만 아는 수준” → “작은 프로그램을 스스로 만들 수 있는 수준” → “실제 프로젝트와 협업에 참여할 수 있는 수준”으로 자연스럽게 성장할 수 있습니다. 각 단계마다 **


 [After] max_tokens=150, stop=['.'] (첫 번째 마침표에서 응답 중단)



 [After] max_tokens=150, temperature=0.8 (창의적인 답변)

